### **TRABAJO PRÁCTICO N° 2: Redes Convolucionales, Detección de Objetos y Redes Recurrentes - Problema 2**
---
 1er cuatrimestre - Año 2026

| **Integrantes**           | **Legajo** |
|---------------------------|------------|
| Martinez Dufour, Caterina | M-7169/2   |
| Grimaldi, Damián Daniel   | G-5977/3   |
| Tapia, Fabrizio           | T-3095/3   |

**Docentes:** *Salvañá, Leandro* - *Fernández, Florencia*

### 1. Descripción del dataset
---

In [6]:
!pip install gdown
!pip install torch torchvision
!pip install ultralytics


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached opencv_python-4.13.0.92-cp37-abi3-win_amd64.whl.metadata (20 kB)
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 1.3/1.3 MB 8.6 MB/s  0:00:00
   ---------------------------------------- 0.0/40.2 MB ? eta -:--:--
   --- ------------------------------------ 3.7/40.2 MB 19.1 MB/s eta 0:00:02
   ------------- -------------------------- 13.4/40.2 MB 32.8 MB/s eta 0:00:01
   ----------------------- ---------------- 23.6/40.2 MB 38.0 MB/s eta 0:00:01
   ---------------------------- ----------- 28.6/40.2 MB 34.4 MB/s eta 0:00:01
   --------------------------------- ------ 33.6/40.2 MB 32.5 MB/s eta 0:00:01
   -------------------------------------- - 38.8/40.2 MB 31.3 MB/s eta 0:00:01
   ---------------------------------------- 40.2/40.2 MB 28.1 MB/s  0:00:01
   ---------------------------------------- 0.0/833.0 kB ? eta -:--:--
   ---------------------------------------- 833.0/833.0 kB 18.2 MB/s  0:00:00
   -------


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Librerías
import pandas as pd
import seaborn as sns
import optuna
import gdown
import zipfile
import os
import random, numpy as np, torch
import matplotlib.pyplot as plt
from PIL import Image
from collections import Counter
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Subset
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import confusion_matrix

d:\FABRO\TUIA\5to Cuatri\Aprendizaje Automatico 2\.venv-1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

### 2. Carga del dataset
---

In [7]:
# ID de tu archivo de Drive
file_id = '1xiiGu2mN5KbKWM1_oRznQEL97hrkSGW_'
url = f'https://drive.google.com/uc?id={file_id}'
archivo_descargado = 'Traffic_Signs_Detection.zip' 

# Descargar el archivo
gdown.download(url, archivo_descargado, quiet=False)
# Descomprimir el archivo en la carpeta
carpeta_destino = 'dataset_problema2'

# Creamos la carpeta si no existe
if not os.path.exists(carpeta_destino):
    os.makedirs(carpeta_destino)

# Extraemos el contenido
try:
    with zipfile.ZipFile(archivo_descargado, 'r') as zip_ref:
        zip_ref.extractall(carpeta_destino)
    # Eliminar el archivo .zip descargado
    os.remove(archivo_descargado) 
    
except zipfile.BadZipFile:
    print("Error.")

Downloading...
From (original): https://drive.google.com/uc?id=1xiiGu2mN5KbKWM1_oRznQEL97hrkSGW_
From (redirected): https://drive.google.com/uc?id=1xiiGu2mN5KbKWM1_oRznQEL97hrkSGW_&confirm=t&uuid=ad85eb58-f15b-4e0c-816d-39a057384f12
To: d:\FABRO\TUIA\5to Cuatri\Aprendizaje Automatico 2\Trabajo Practico 2\problema_2\Traffic_Signs_Detection.zip
100%|██████████| 105M/105M [00:03<00:00, 29.6MB/s] 


### 3. Preparación de datos
---